# Walk-Forward Backtest ончейн стратегий на 1d

### Setup + Config

In [ ]:
import logging
import subprocess
import sys
from pathlib import Path

import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

from src.artifacts import safe_write_parquet
from src import report as report_mod

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("runner_onchain_nb")

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").exists():
    raise FileNotFoundError(f"Cannot locate project root from cwd={Path.cwd()}")

SCRIPT_PATH = PROJECT_ROOT / "run_runner_onchain.py"
RESULTS_DIR = PROJECT_ROOT / "results" / "runner_onchain"
FIGURES_DIR = PROJECT_ROOT / "figures" / "runner_onchain"
PPY = 365
TOP_K_PLOTS = 3

if not SCRIPT_PATH.exists():
    raise FileNotFoundError(f"Missing script: {SCRIPT_PATH}")

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SCRIPT_PATH:", SCRIPT_PATH)
print("RESULTS_DIR:", RESULTS_DIR)
print("FIGURES_DIR:", FIGURES_DIR)


PROJECT_ROOT: c:\Users\prosh\OneDrive\Рабочий стол\git\diploma
SCRIPT_PATH: c:\Users\prosh\OneDrive\Рабочий стол\git\diploma\run_runner_onchain.py
RESULTS_DIR: c:\Users\prosh\OneDrive\Рабочий стол\git\diploma\results\runner_onchain
FIGURES_DIR: c:\Users\prosh\OneDrive\Рабочий стол\git\diploma\figures\runner_onchain


## Прогон WF

In [ ]:
import time

cmd = [sys.executable, "-W", "ignore::UserWarning", "-u", str(SCRIPT_PATH), "--mp-workers", "4"] 
print("Running:", " ".join(cmd))

t0 = time.time()
proc = subprocess.Popen(
    cmd,
    cwd=str(PROJECT_ROOT),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in proc.stdout:
    print(line, end="")

rc = proc.wait()
print(f"\nReturn code: {rc} | elapsed: {(time.time() - t0)/60:.1f} min")

if rc != 0:
    raise RuntimeError(f"WF script failed with return code={rc}")


Running: c:\Users\prosh\OneDrive\Рабочий стол\git\diploma\.venv\Scripts\python.exe -W ignore::UserWarning -u c:\Users\prosh\OneDrive\Рабочий стол\git\diploma\run_runner_onchain.py --mp-workers 4
2026-04-07 13:25:36,322 | INFO | Configured backtesting.Pool = multiprocessing.Pool(processes=4).
2026-04-07 13:25:36,323 | INFO | PROJECT_ROOT=c:\Users\prosh\OneDrive\Р Р°Р±РѕС‡РёР№ СЃС‚РѕР»\git\diploma
2026-04-07 13:25:36,333 | INFO | On-chain strategies: ['R1OC:RSI_MR_Onchain', 'R2OC:Bollinger_MR_Onchain', 'R3OC:ZScore_MR_Onchain', 'S1OC:MAFilter_RSI_MR_Onchain', 'S2OC:MA200Filter_Bollinger_MR_Onchain']
2026-04-07 13:25:36,333 | INFO | Benchmark: BENCH:BuyHold
2026-04-07 13:25:36,385 | INFO | BTCUSDT loaded: n=3119 | 2017-09-16 .. 2026-03-31
2026-04-07 13:25:36,425 | INFO | ETHUSDT loaded: n=3119 | 2017-09-16 .. 2026-03-31
2026-04-07 13:25:36,463 | INFO | XRPUSDT loaded: n=2859 | 2018-06-03 .. 2026-03-31
2026-04-07 13:25:36,500 | INFO | DOGEUSDT loaded: n=2432 | 2019-08-04 .. 2026-03-31
2026

In [ ]:
folds_best = pd.read_parquet(RESULTS_DIR / "folds_best.parquet")
returns_oos = pd.read_parquet(RESULTS_DIR / "returns_oos.parquet")
bench_oos = pd.read_parquet(RESULTS_DIR / "bench_returns_oos.parquet")
bench_folds = pd.read_parquet(RESULTS_DIR / "bench_folds.parquet")


def _ensure_date_col(df_in: pd.DataFrame) -> pd.DataFrame:
    df = df_in.copy()
    if "date" not in df.columns:
        if "datetime_utc" in df.columns:
            df = df.rename(columns={"datetime_utc": "date"})
        elif "index" in df.columns:
            df = df.rename(columns={"index": "date"})
        else:
            raise KeyError(f"No date column in dataframe. Columns={list(df.columns)[:30]}")
    df["date"] = pd.to_datetime(df["date"])
    return df


def _build_benchmark_comparison(folds_best_df: pd.DataFrame, bench_folds_df: pd.DataFrame):
    if bench_folds_df.empty or folds_best_df.empty:
        return pd.DataFrame(), pd.DataFrame()

    merged = folds_best_df.merge(
        bench_folds_df[["symbol", "cost", "fold_id", "bench_calmar", "bench_total_return"]],
        on=["symbol", "cost", "fold_id"],
        how="left",
        validate="many_to_one",
    )
    merged["beat_bench_calmar"] = (merged["oos_calmar"] > merged["bench_calmar"]).astype(int)
    merged["beat_bench_total_return"] = (merged["oos_total_return"] > merged["bench_total_return"]).astype(int)
    merged["excess_calmar"] = merged["oos_calmar"] - merged["bench_calmar"]
    merged["excess_total_return"] = merged["oos_total_return"] - merged["bench_total_return"]

    beat = (
        merged.groupby(["symbol", "cost", "strategy_id"])
        .agg(
            folds=("fold_id", "nunique"),
            beat_calmar_share=("beat_bench_calmar", "mean"),
            beat_return_share=("beat_bench_total_return", "mean"),
            calmar_median=("oos_calmar", "median"),
            calmar_worst=("oos_calmar", "min"),
        )
        .reset_index()
    )
    excess = (
        merged.groupby(["symbol", "cost", "strategy_id"])
        .agg(
            folds=("fold_id", "nunique"),
            excess_calmar_median=("excess_calmar", "median"),
            excess_return_median=("excess_total_return", "median"),
        )
        .reset_index()
    )
    return beat, excess


def _plot_top_equity_drawdown(returns_oos_df: pd.DataFrame, bench_oos_df: pd.DataFrame, stitched_df: pd.DataFrame, top_k: int):
    rets = _ensure_date_col(returns_oos_df)
    bench = _ensure_date_col(bench_oos_df)

    if stitched_df.empty:
        return

    ranked = stitched_df.sort_values(["symbol", "cost", "Calmar"], ascending=[True, True, False])
    bench_id = "BENCH:BuyHold"
    for (sym, cost), group in ranked.groupby(["symbol", "cost"]):
        top = group.head(int(top_k))
        for _, row in top.iterrows():
            sid = str(row["strategy_id"])
            if sid.startswith("BENCH:"):
                continue

            r = rets[(rets["symbol"] == sym) & (rets["cost"] == cost) & (rets["strategy_id"] == sid)].copy()
            b = bench[(bench["symbol"] == sym) & (bench["cost"] == cost) & (bench["strategy_id"] == bench_id)].copy()
            if r.empty or b.empty:
                continue

            rs = r.set_index("date")["ret"].astype(float)
            bs = b.set_index("date")["ret"].astype(float)
            report_mod.plot_equity_and_drawdown(
                returns=rs,
                bench_returns=bs,
                title=f"{sid} vs Buy&Hold | {sym} | {cost}",
                out_path=FIGURES_DIR / f"equity_dd__{sym}__{cost}__{sid.replace(':', '_')}.png",
            )


leaderboard = report_mod.leaderboard_by_strategy(folds_best)
stitched = report_mod.stitched_metrics(returns_oos, periods_per_year=PPY)
bench_stitched = report_mod.stitched_metrics(bench_oos, periods_per_year=PPY)
beat, excess = _build_benchmark_comparison(folds_best, bench_folds)

safe_write_parquet(leaderboard, RESULTS_DIR / "leaderboard.parquet")
safe_write_parquet(stitched, RESULTS_DIR / "stitched_metrics.parquet")
safe_write_parquet(bench_stitched, RESULTS_DIR / "bench_stitched_metrics.parquet")
safe_write_parquet(beat, RESULTS_DIR / "beat_benchmark_summary.parquet")
safe_write_parquet(excess, RESULTS_DIR / "excess_vs_benchmark_summary.parquet")

_plot_top_equity_drawdown(returns_oos, bench_oos, stitched, TOP_K_PLOTS)

print("Saved report artifacts to", RESULTS_DIR)
print("Saved figures to", FIGURES_DIR)


Saved report artifacts to c:\Users\prosh\OneDrive\Рабочий стол\git\diploma\results\runner_onchain
Saved figures to c:\Users\prosh\OneDrive\Рабочий стол\git\diploma\figures\runner_onchain


In [ ]:
def _shape(path: Path):
    if not path.exists():
        return None
    return pd.read_parquet(path).shape

keys = [
    "folds_best.parquet",
    "returns_oos.parquet",
    "bench_returns_oos.parquet",
    "bench_folds.parquet",
    "leaderboard.parquet",
    "stitched_metrics.parquet",
    "bench_stitched_metrics.parquet",
    "beat_benchmark_summary.parquet",
    "excess_vs_benchmark_summary.parquet",
]

shape_df = pd.DataFrame(
    [{"file": k, "shape": _shape(RESULTS_DIR / k)} for k in keys]
)
display(shape_df)

display(pd.read_parquet(RESULTS_DIR / "stitched_metrics.parquet").head(20))


,file,shape
0,folds_best.parquet,"(570, 23)"
1,returns_oos.parquet,"(104070, 5)"
2,bench_returns_oos.parquet,"(20814, 5)"
3,bench_folds.parquet,"(114, 13)"
4,leaderboard.parquet,"(60, 36)"
5,stitched_metrics.parquet,"(60, 12)"
6,bench_stitched_metrics.parquet,"(12, 12)"
7,beat_benchmark_summary.parquet,"(60, 8)"
8,excess_vs_benchmark_summary.parquet,"(60, 6)"


,symbol,cost,strategy_id,n_bars,Total Return,CAGR,Ann. Vol,Sharpe,Sortino,MaxDD,Calmar,CDaR_95
0,BTCUSDT,Base,S1OC:MAFilter_RSI_MR_Onchain,2007,0.339518,0.054599,0.153931,0.422404,0.618120,-0.227380,0.240121,0.151384
1,BTCUSDT,Base,S2OC:MA200Filter_Bollinger_MR_Onchain,2007,0.227882,0.038041,0.156611,0.316650,0.460714,-0.227380,0.167300,0.151048
2,BTCUSDT,Base,R3OC:ZScore_MR_Onchain,2007,0.265781,0.043795,0.266399,0.294423,0.435333,-0.375023,0.116780,0.328694
3,BTCUSDT,Base,R2OC:Bollinger_MR_Onchain,2007,0.194356,0.032828,0.282386,0.255690,0.381094,-0.375023,0.087535,0.320331
4,BTCUSDT,Base,R1OC:RSI_MR_Onchain,2007,-0.015831,-0.002898,0.284419,0.131994,0.196346,-0.447579,-0.006474,0.405006
5,BTCUSDT,High,S1OC:MAFilter_RSI_MR_Onchain,2007,0.315361,0.051114,0.153894,0.400976,0.585313,-0.227380,0.224796,0.151796
6,BTCUSDT,High,S2OC:MA200Filter_Bollinger_MR_Onchain,2007,0.204172,0.034366,0.156599,0.294027,0.426957,-0.227380,0.151140,0.151477
7,BTCUSDT,High,R3OC:ZScore_MR_Onchain,2007,0.228497,0.038135,0.266380,0.274041,0.404542,-0.379879,0.100388,0.335447
8,BTCUSDT,High,R2OC:Bollinger_MR_Onchain,2007,0.157671,0.026984,0.282368,0.235619,0.350721,-0.379879,0.071034,0.326830
9,BTCUSDT,High,R1OC:RSI_MR_Onchain,2007,-0.049297,-0.009152,0.284546,0.109987,0.163322,-0.457270,-0.020014,0.415824
